In [11]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

TensorFlow: 2.21.0
GPU: []


In [ ]:
# Reload CIFAR-10
(x_train, y_train), (x_test, y_test) = (
    tf.keras.datasets.cifar10.load_data()
)

# Split training and validation data
x_val, y_val = x_train[:5000], y_train[:5000]
x_train, y_train = x_train[5000:], y_train[5000:]

# Preprocessing function
def prepare_image(image, label):
    image = tf.cast(image, tf.float32)
    image = tf.image.resize(image, (96, 96))
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    return image, label

# Create pipelines
train_ds = tf.data.Dataset.from_tensor_slices(
    (x_train, y_train)
)

val_ds = tf.data.Dataset.from_tensor_slices(
    (x_val, y_val)
)

train_ds = (
    train_ds.shuffle(5000)
    .map(prepare_image, num_parallel_calls=1)
    .batch(32)
    .prefetch(1)
)

val_ds = (
    val_ds.map(prepare_image, num_parallel_calls=1)
    .batch(32)
    .prefetch(1)
)

print("Datasets prepared!")

167436288/170498071 ━━━━━━━━━━━━━━━━━━━━ 15s 5us/step

In [ ]:
(x_train, y_train), (x_test, y_test) = (
    tf.keras.datasets.cifar10.load_data()
)

print("Training images:", x_train.shape)
print("Training labels:", y_train.shape)
print("Testing images:", x_test.shape)
print("Testing labels:", y_test.shape)

In [ ]:
class_names = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

plt.figure(figsize=(12, 6))

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i][0]])
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(96, 96, 3)
)

In [6]:
base_model.trainable = False

In [7]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(96, 96, 3)),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10, activation='softmax')
])

In [8]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_96             │ (None, 3, 3, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,270,794 (8.66 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [9]:

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [10]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

Epoch 1/5
 169/1407 ━━━━━━━━━━━━━━━━━━━━ 4:18 208ms/step - accuracy: 0.4782 - loss: 1.6096

KeyboardInterrupt: 

In [ ]:
# Plot training and validation accuracy

plt.figure(figsize=(10, 5))

plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')

plt.title('Training vs Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')

plt.title('Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
test_ds = tf.data.Dataset.from_tensor_slices(
    (x_test, y_test)
)

test_ds = (
    test_ds
    .map(prepare_image, num_parallel_calls=1)
    .batch(32)
    .prefetch(1)
)

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)

print("Test Accuracy:", test_accuracy)
print("Test Loss:", test_loss)

In [ ]:
y_pred_probs = model.predict(test_ds)

y_pred = np.argmax(y_pred_probs, axis=1)

y_true = y_test.flatten()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(11, 9))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

disp.plot(cmap='Blues', xticks_rotation=45)
plt.title("CIFAR-10 Confusion Matrix")
plt.show()

In [ ]:
model.save("mobilenetv2_frozen.keras")

In [ ]:
# Allow fine-tuning
base_model.trainable = True

# Freeze all but the last 30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Keep Batch Normalisation layers frozen
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

print("Total layers:", len(base_model.layers))
print("Trainable layers:",
      sum(layer.trainable for layer in base_model.layers))

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.00001
    ),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
fine_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3
)

In [ ]:
model.save("mobilenetv2_finetuned.keras")

In [ ]:
#evaluate the fine tuned model

fine_test_loss, fine_test_accuracy = model.evaluate(test_ds)

print("Fine-tuned Test Accuracy:", fine_test_accuracy)
print("Fine-tuned Test Loss:", fine_test_loss)

In [ ]:
tf.keras.utils.plot_model(
    model,
    show_shapes=True,
    show_layer_names=True,
    expand_nested=True,
    to_file="model_architecture.png"
)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))

In [15]:
print("Model:", model)
print("Test dataset:", test_ds)
print("GPU:", tf.config.list_physical_devices('GPU'))

Model: <Sequential name=sequential, built=True>
Test dataset: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 96, 96, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 1), dtype=tf.uint8, name=None))>
GPU: []


In [14]:
test_ds = tf.data.Dataset.from_tensor_slices(
    (x_test, y_test)
)

test_ds = (
    test_ds
    .map(prepare_image, num_parallel_calls=1)
    .batch(32)
    .prefetch(1)
)

print("Test dataset recreated!")

Test dataset recreated!


In [16]:
#Evaluate the fine-tuned model
fine_test_loss, fine_test_accuracy = model.evaluate(test_ds)

print("Fine-tuned Test Accuracy:", fine_test_accuracy)
print("Fine-tuned Test Loss:", fine_test_loss)

313/313 ━━━━━━━━━━━━━━━━━━━━ 76s 234ms/step - accuracy: 0.7870 - loss: 0.6193
Fine-tuned Test Accuracy: 0.7870000004768372
Fine-tuned Test Loss: 0.6193257570266724


In [17]:
import os

print(os.path.exists("mobilenetv2_frozen.keras"))
print(os.path.exists("mobilenetv2_finetuned.keras"))

False
False


In [18]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [19]:
model.save(
    '/content/drive/MyDrive/mobilenetv2_finetuned.keras'
)

In [20]:
import os

path = '/content/drive/MyDrive/mobilenetv2_finetuned.keras'

print("Model saved:", os.path.exists(path))
print("File size (MB):", round(os.path.getsize(path)/1024**2, 2))

Model saved: True
File size (MB): 9.32


In [21]:
from sklearn.metrics import classification_report

# Generate predictions
y_pred_probs = model.predict(test_ds)

# Convert probabilities into predicted classes
y_pred = np.argmax(y_pred_probs, axis=1)

# Actual labels
y_true = y_test.flatten()

# Generate evaluation report
print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))

313/313 ━━━━━━━━━━━━━━━━━━━━ 82s 253ms/step
              precision    recall  f1-score   support

    airplane       0.83      0.73      0.78      1000
  automobile       0.89      0.91      0.90      1000
        bird       0.70      0.78      0.74      1000
         cat       0.68      0.56      0.62      1000
        deer       0.71      0.80      0.75      1000
         dog       0.69      0.77      0.73      1000
        frog       0.89      0.76      0.82      1000
       horse       0.83      0.78      0.81      1000
        ship       0.77      0.94      0.84      1000
       truck       0.92      0.84      0.88      1000

    accuracy                           0.79     10000
   macro avg       0.79      0.79      0.79     10000
weighted avg       0.79      0.79      0.79     10000



In [22]:
print("Image shape:", x_test.shape)
print("Test dataset:", test_ds.element_spec)
print("Image preprocessing:", prepare_image)
print("Batch size: 32")

Image shape: (10000, 32, 32, 3)
Test dataset: (TensorSpec(shape=(None, 96, 96, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 1), dtype=tf.uint8, name=None))
Image preprocessing: <function prepare_image at 0x7b990a72fd80>
Batch size: 32
